In [1]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


### 1. 데이터 탐색

판다스를 이용하여 데이터를 탐색하세요.

1. 전체 행 개수와 열 개수를 출력하세요.
2. 상위 5행을 출력하세요.
3. 각 열의 데이터 타입을 확인하세요.
4. `describe()`로 통계 정보를 확인하세요.
5. `target`의 값별 데이터 개수를 출력하세요.
6. 양성(1)과 음성(0)의 비율을 백분율로 출력하세요.

In [3]:
# 1. 전체 행 개수와 열 개수를 출력하세요.
print(df.shape)

(768, 9)


In [4]:
# 2. 상위 5행을 출력하세요.
print(df.head())

   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1
3     1    89    66    23    94  28.1  0.167   21       0
4     0   137    40    35   168  43.1  2.288   33       1


In [5]:
# 3. 각 열의 데이터 타입을 확인하세요.
df.dtypes

preg        int64
plas        int64
pres        int64
skin        int64
insu        int64
mass      float64
pedi      float64
age         int64
target      int64
dtype: object

In [6]:
# 4. describe()로 통계 정보를 확인하세요.
df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age,target
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [7]:
# 5. target의 값별 데이터 개수를 출력하세요.
print(df.groupby("target").size())

target
0    500
1    268
dtype: int64


In [8]:
# 6. 양성(1)과 음성(0)의 비율을 백분율로 출력하세요.
print((df["target"] == 1).mean() * 100)
print((df["target"] == 0).mean() * 100)

34.89583333333333
65.10416666666666


**분석 질문**

1. 전체 데이터에서 양성(1)의 비율은 몇 %인가요?
2. 입력 특성 중 숫자 범위가 가장 큰 특성과 가장 작은 특성은 무엇인가요?
3. 특성마다 숫자 범위가 다른 것이 KNN에 어떤 영향을 줄 수 있을까요?

답
1. 약 34.9%입니다.
2. 숫자 범위가 가장 큰 특성은 insu이고 가장 작은 특성은 pedi입니다.
3. 숫자 범위가 큰 특성이 계산에 과대 집계될 수 있습니다.

### 2. 결측치 탐색

데이터에서 누락된 측정값을 찾아보세요.

1. `isna().sum()`으로 결측치 개수를 출력하세요.
2. 준비 셀에서 안내한 5개 열의 0 개수를 각각 출력하세요.
3. 해당 5개 열에서 0을 `NaN`으로 변경하세요.
4. 다시 `isna().sum()`으로 결측치 개수를 확인하세요.
5. 결측치가 하나라도 포함된 행의 개수를 출력하세요.

**주의:** 이 단계에서는 결측치를 삭제하거나 다른 값으로 채우지 않습니다.

In [9]:
# 1. isna().sum()으로 결측치 개수를 출력하세요.
print(df.isna().sum())

preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64


In [10]:
# 2. 준비 셀에서 안내한 5개 열의 0 개수를 각각 출력하세요.
print((df["plas"] == 0).sum())
print((df["pres"] == 0).sum())
print((df["skin"] == 0).sum())
print((df["insu"] == 0).sum())
print((df["mass"] == 0).sum())

5
35
227
374
11


In [11]:
# 3. 해당 5개 열에서 0을 NaN으로 변경하세요.
import numpy as np
df[["plas" , "pres" , "skin" , "insu" , "mass"]] = df[["plas" , "pres" , "skin" , "insu" , "mass"]].replace(0 , np.nan)

In [12]:
# 4. 다시 isna().sum()으로 결측치 개수를 확인하세요.
df.isna().sum()

preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64

In [13]:
# 5. 결측치가 하나라도 포함된 행의 개수를 출력하세요.
print(df.isna().any(axis=1).sum())

376


**분석 질문**

1. 처음 `isna()`를 실행했을 때 결측치가 발견되지 않은 이유는 무엇인가요?
2. 모든 열의 0을 `NaN`으로 변경하면 안 되는 이유는 무엇인가요?

답
1. 0이라는 값이 있었기 때문입니다.
2. preg, target 값이 0인 것은 정상적인 값으로 NaN으로 바꾸면 안 되기 때문입니다.

### 3. 특성과 정답 분리 및 데이터 분할

머신러닝을 학습하기 위한 데이터를 준비하세요.

1. `target`을 제외한 열들을 `X`에 저장하세요.
2. `target`을 `y`에 저장하세요.
3. `X`와 `y`의 shape를 출력하세요.
4. `train_test_split()`으로 데이터를 분리하세요.

In [14]:
# 1. target을 제외한 열들을 X에 저장하세요.
X = df[["preg" , "plas" , "pres" , "skin" , "insu" , "mass" , "pedi" , "age"]]

In [15]:
# 2. target을 y에 저장하세요.
y = df["target"]

In [16]:
# 3. X와 y의 shape를 출력하세요.
print(X.shape)
print(y.shape)

(768, 8)
(768,)


In [17]:
# 4. train_test_split()으로 데이터를 분리하세요.
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)

**분할 조건**

- 학습 데이터 80%
- 테스트 데이터 20%
- `random_state=42`
- `stratify=y`
1. `X_train`, `X_test`, `y_train`, `y_test`의 shape를 출력하세요.
2. 학습 데이터와 테스트 데이터의 양성·음성 개수를 각각 출력하세요.

In [18]:
# 1. X_train, X_test, y_train, y_test의 shape를 출력하세요.
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(614, 8)
(154, 8)
(614,)
(154,)


In [19]:
# 2. 학습 데이터와 테스트 데이터의 양성·음성 개수를 각각 출력하세요.
print((y_train == 1).sum())
print((y_train == 0).sum())
print((y_test == 1).sum())
print((y_test == 0).sum())

214
400
54
100


**분석 질문**

1. 학습 데이터와 테스트 데이터를 나누는 이유는 무엇인가요?
2. `stratify=y`를 사용하는 이유는 무엇인가요?
3. `random_state`를 지정하면 어떤 장점이 있나요?

답
1. 학습하지 않은 데이터로 테스트를 해야 모델이 잘 작동하는지 파악할 수 있기 때문입니다.
2. target 값의 비율에 따라 데이터가 분리됩니다.

### 4. 결측치 처리

3번에서 분리한 학습 데이터와 테스트 데이터의 결측치를 처리하세요.

**조건**

- 결측치는 각 특성의 중앙값(`median`)으로 채우세요.
- 중앙값은 학습 데이터에서만 계산하세요.
- 테스트 데이터에는 학습 데이터에서 계산한 중앙값을 사용하세요.
- 원본 데이터는 유지하고 결측치를 처리한 데이터를 별도의 변수에 저장하세요.

다음 작업을 수행하세요.

1. 학습 데이터에서 각 특성의 중앙값을 계산하세요.
2. 학습 데이터의 결측치를 중앙값으로 채우세요.
3. 테스트 데이터의 결측치도 동일한 중앙값으로 채우세요.
4. 처리 후 학습 데이터와 테스트 데이터의 결측치 개수를 `isna().sum()`으로 출력하세요.
5. 처리 전후 데이터의 행 개수를 비교하세요.

In [20]:
# 1. 학습 데이터에서 각 특성의 중앙값을 계산하세요.
plas_med = X_train["plas"].median()
pres_med = X_train["pres"].median()
skin_med = X_train["skin"].median()
insu_med = X_train["insu"].median()
mass_med = X_train["mass"].median()

In [21]:
# 2. 학습 데이터의 결측치를 중앙값으로 채우세요.
X_train_cp = X_train.copy()
X_train_cp["plas"] = X_train_cp["plas"].fillna(plas_med)
X_train_cp["pres"] = X_train_cp["pres"].fillna(pres_med)
X_train_cp["skin"] = X_train_cp["skin"].fillna(skin_med)
X_train_cp["insu"] = X_train_cp["insu"].fillna(insu_med)
X_train_cp["mass"] = X_train_cp["mass"].fillna(mass_med)

In [22]:
# 3. 테스트 데이터의 결측치도 동일한 중앙값으로 채우세요.
X_test_cp = X_test.copy()
X_test_cp["plas"] = X_test_cp["plas"].fillna(plas_med)
X_test_cp["pres"] = X_test_cp["pres"].fillna(pres_med)
X_test_cp["skin"] = X_test_cp["skin"].fillna(skin_med)
X_test_cp["insu"] = X_test_cp["insu"].fillna(insu_med)
X_test_cp["mass"] = X_test_cp["mass"].fillna(mass_med)

In [23]:
# 4. 처리 후 학습 데이터와 테스트 데이터의 결측치 개수를 isna().sum()으로 출력하세요.
print(X_train_cp.isna().sum())
print()
print(X_test_cp.isna().sum())

preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64

preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64


In [24]:
# 5. 처리 전후 데이터의 행 개수를 비교하세요.
X_train.shape[0] , X_train_cp.shape[0]

(614, 614)

**분석 질문**

1. 결측치를 삭제하지 않고 중앙값으로 채우면 어떤 장점이 있나요?
2. 테스트 데이터의 중앙값을 따로 계산해서 사용하면 안 되는 이유는 무엇인가요?

답
1. 모든 데이터를 활용할 수 있습니다.
2. 테스트 데이터의 중앙값을 계산하면 모델이 테스트를 하기도 전에 정보를 미리 알 수 있습니다.

### 5. 첫 번째 KNN 학습

결측치 처리가 완료된 데이터를 이용하여 KNN을 학습하세요.

**조건**

- `KNeighborsClassifier` 사용
- `n_neighbors=5`
- 스케일링하지 않은 데이터 사용
- 4번에서 결측치를 처리한 학습·테스트 데이터 사용

다음 작업을 수행하세요.

1. KNN 객체를 생성하세요.
2. `fit()`으로 학습하세요.
3. 학습 정확도를 출력하세요.
4. 테스트 정확도를 출력하세요.
5. `predict()`로 테스트 데이터의 결과를 예측하세요.
6. 예측 결과와 실제 정답을 각각 앞 10개씩 출력하세요.

In [25]:
# 1. KNN 객체를 생성하세요.
model = KNeighborsClassifier(n_neighbors=5)

In [26]:
# 2. fit()으로 학습하세요.
model.fit(X_train_cp , y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [27]:
# 3. 학습 정확도를 출력하세요.
print(model.score(X_train_cp , y_train))

0.8013029315960912


In [28]:
# 4. 테스트 정확도를 출력하세요.
print(model.score(X_test_cp , y_test))

0.6753246753246753


In [29]:
# 5. predict()로 테스트 데이터의 결과를 예측하세요.
model.predict(X_test_cp)

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0])

In [30]:
# 6. 예측 결과와 실제 정답을 각각 앞 10개씩 출력하세요.
print(model.predict(X_test_cp)[:10])
print(y_test.head(10).tolist())

[1 0 0 0 0 0 0 1 0 1]
[0, 0, 0, 1, 0, 0, 1, 1, 0, 0]


**분석 질문**

1. KNN에서 k=5는 어떤 의미인가요?
2. 학습 정확도와 테스트 정확도는 각각 무엇을 의미하나요?
3. 학습 정확도와 테스트 정확도에 차이가 발생하는 이유는 무엇인가요?

답
1. 가장 가까운 5개 중에서 다수결로 정한다는 뜻입니다.
2. 학습 정확도는 모델이 학습 시 정답을 맞힌 비율이고, 테스트 정확도는 모델이 새 데이터에서 정답을 맞힌 비율입니다.
3. 테스트 정확도가 크게 낮은 것으로 보아 과적합일 가능성이 있습니다.

### 6. KNN 성능 평가

5번에서 학습한 KNN의 예측 결과를 평가하세요.

다음 항목을 출력하세요.

1. 혼동행렬 (`confusion_matrix`)
2. 정확도 (`accuracy_score`)
3. 정밀도 (`precision_score`)
4. 재현율 (`recall_score`)
5. F1 점수 (`f1_score`)
6. `classification_report`

이번 데이터는 이진 분류이므로 `0`은 음성, `1`은 양성입니다. 정밀도와 재현율은 양성(1)을 기준으로 계산하세요.

In [31]:
# 1. 혼동행렬 (confusion_matrix)
pred = model.predict(X_test_cp)
print(confusion_matrix(y_test , pred))

[[77 23]
 [27 27]]


In [32]:
# 2. 정확도 (accuracy_score)
print(accuracy_score(y_test , pred))

0.6753246753246753


In [33]:
# 3. 정밀도 (precision_score)
print(precision_score(y_test , pred))

0.54


In [34]:
# 4. 재현율 (recall_score)
print(recall_score(y_test , pred))

0.5


In [35]:
# 5. F1 점수 (f1_score)
print(f1_score(y_test , pred))

0.5192307692307693


In [36]:
# 6. classification_report
print(classification_report(y_test , pred))

              precision    recall  f1-score   support

           0       0.74      0.77      0.75       100
           1       0.54      0.50      0.52        54

    accuracy                           0.68       154
   macro avg       0.64      0.64      0.64       154
weighted avg       0.67      0.68      0.67       154



**분석 질문**

1. 실제 양성인데 음성으로 잘못 예측한 데이터(FN)는 몇 개입니까?
2. 실제 음성인데 양성으로 잘못 예측한 데이터(FP)는 몇 개입니까?
3. 정확도는 높지만 재현율은 낮은 상황이 발생할 수 있나요?
4. 당뇨병 양성을 놓치는 것을 줄이려면 어떤 평가 지표를 중요하게 봐야 할까요?

답
1. 27개입니다.
2. 23개입니다.
3. 기준을 보수적으로 설정하면 정확도는 높지만 재현율은 낮은 상황이 발생할 수 있습니다.
4. 재현율을 중요하게 봐야 합니다.

### 7. k값에 따른 성능 비교

KNN의 k값을 변경하면서 학습 정확도와 테스트 정확도가 어떻게 변하는지 확인하세요.

실험할 k값:

`1, 3, 5, 7, 10, 15, 20, 30`

**조건**

- 반복문을 사용하세요.
- 스케일링하지 않은 데이터를 사용하세요.
- 4번에서 결측치를 처리한 데이터를 사용하세요.
- k값 외에는 동일한 조건을 유지하세요.

각 k값의 학습 정확도와 테스트 정확도를 출력하고 다음 표를 완성하세요.

마크다운 또는 코드셀에 주석으로 적어주시면 됩니다.
e.g. 
1 / 0.91 / 0.93 
3 / 0.92 / 0.94
이런식으로 ,,,,


In [37]:
n_neighbors = [1 , 3 , 5 , 7 , 10 , 15 , 20 , 30]
for n in n_neighbors:
    model = KNeighborsClassifier(n_neighbors=n)
    model.fit(X_train_cp , y_train)
    train_sco = model.score(X_train_cp , y_train)
    test_sco = model.score(X_test_cp , y_test)
    print(round(train_sco , 2) , round(test_sco , 2))

1.0 0.68
0.84 0.68
0.8 0.68
0.79 0.68
0.8 0.67
0.79 0.66
0.79 0.72
0.77 0.72


### 1 / 1.0 / 0.68
### 3 / 0.84 / 0.68
### 5 / 0.8 / 0.68
### 7 / 0.79 / 0.68
### 10 / 0.8 / 0.67
### 15 / 0.79 / 0.66
### 20 / 0.79 / 0.72
### 30 / 0.77 / 0.72

**분석 질문**

1. 학습 정확도가 가장 높은 k값은 무엇인가요?
2. 테스트 정확도가 가장 높은 k값은 무엇인가요?
3. k가 작을 때 과대적합이 발생하기 쉬운 이유는 무엇인가요?
4. k가 지나치게 커지면 과소적합이 발생할 수 있는 이유는 무엇인가요?
5. 학습 정확도만 보고 k값을 선택해도 될까요?

답
1. 1입니다.
2. 20과 30입니다.
3. 가장 가까운 소수만 고려하여 경계선이 복잡해지기 때문입니다.
4. 지나치게 많은 수를 고려하여 경계선이 단순해지기 때문입니다.
5. 안 됩니다. 재현율도 함께 고려해야 합니다.

### 8. StandardScaler 적용

이번에는 `StandardScaler`로 특성을 표준화한 뒤 동일한 KNN을 학습하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `StandardScaler` 사용
- 스케일러는 학습 데이터에만 `fit()`
- 학습 데이터와 테스트 데이터는 같은 스케일러로 변환
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `StandardScaler` 객체를 생성하세요.
2. 학습 데이터로 `fit()` 하세요.
3. 학습 데이터와 테스트 데이터를 각각 `transform()` 하세요.
4. 변환된 학습 데이터를 DataFrame으로 만들고 `describe()`를 출력하세요.
5. 변환된 데이터로 KNN(k=5)을 학습하세요.
6. 학습 정확도와 테스트 정확도를 출력하세요.
7. 재현율을 출력하세요.

In [38]:
# 1. StandardScaler 객체를 생성하세요.
scaler = StandardScaler()

In [39]:
# 2. 학습 데이터로 fit() 하세요.
scaler.fit(X_train_cp)

StandardScaler()

In [40]:
# 3. 학습 데이터와 테스트 데이터를 각각 transform() 하세요.
X_train_scaled = scaler.transform(X_train_cp)
X_test_scaled = scaler.transform(X_test_cp)

In [41]:
# 4. 변환된 학습 데이터를 DataFrame으로 만들고 describe()를 출력하세요.
X_train_scaled_df = pd.DataFrame(X_train_scaled , columns=X.columns)
print(X_train_scaled_df.describe())

               preg          plas          pres          skin          insu  \
count  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02   
mean  -6.943414e-17 -1.099374e-16  3.095606e-16 -3.471707e-17 -4.339634e-18   
std    1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00   
min   -1.153338e+00 -2.190541e+00 -3.924957e+00 -2.480958e+00 -1.559139e+00   
25%   -8.513551e-01 -7.562208e-01 -6.636760e-01 -4.549828e-01 -2.249692e-01   
50%   -2.473897e-01 -1.558076e-01 -1.141979e-02 -4.766137e-03 -1.614373e-01   
75%    6.585583e-01  6.113872e-01  6.408364e-01  3.328964e-01 -9.790537e-02   
max    3.980368e+00  2.579408e+00  4.065181e+00  7.874025e+00  7.703814e+00   

               mass          pedi           age  
count  6.140000e+02  6.140000e+02  6.140000e+02  
mean   1.649061e-16 -1.099374e-16 -1.084908e-16  
std    1.000815e+00  1.000815e+00  1.000815e+00  
min   -2.089621e+00 -1.192094e+00 -1.045895e+00  
25%   -7.073644e-01 -7.042617e-01 -7.9

In [42]:
# 5. 변환된 데이터로 KNN(k=5)을 학습하세요.
model_scaled = KNeighborsClassifier(n_neighbors=5)
model_scaled.fit(X_train_scaled , y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [43]:
# 6. 학습 정확도와 테스트 정확도를 출력하세요.
print(model_scaled.score(X_train_scaled , y_train))
print(model_scaled.score(X_test_scaled , y_test))

0.8289902280130294
0.7532467532467533


In [44]:
# 7. 재현율을 출력하세요.
pred_scaled = model_scaled.predict(X_test_scaled)
print(recall_score(y_test , pred_scaled))

0.6111111111111112


**분석 질문**

1. 표준화 후 각 특성의 평균과 표준편차는 대략 얼마인가요?
2. 5번에서 구한 스케일링 전 정확도와 비교하면 어떤 차이가 있나요?
3. 모델의 알고리즘과 k값은 그대로인데 정확도가 달라지는 이유는 무엇인가요?

답
1. 평균은 4.701270136849196e-18이고, 표준편차는 1.000815입니다.
2. 0.6753246753246753에서 0.7532467532467533로 상승했습니다.
3. 스케일링으로 각 특성의 범위가 동일해졌기 때문입니다.

### 9. MinMaxScaler 적용 및 비교

이번에는 `MinMaxScaler`를 적용하고 KNN의 성능을 확인하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `MinMaxScaler` 사용
- 학습 데이터에만 `fit()`
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `MinMaxScaler`로 학습·테스트 데이터를 변환하세요.
2. 변환된 학습 데이터의 `describe()`를 출력하세요.
3. 변환된 데이터로 KNN(k=5)을 학습하세요.
4. 학습 정확도와 테스트 정확도를 출력하세요.
5. 재현율을 출력하세요.
6. 세 가지 전처리 방법의 성능을 비교하세요.

In [45]:
# 1. MinMaxScaler로 학습·테스트 데이터를 변환하세요.
minmax = MinMaxScaler()
minmax.fit(X_train)
X_train_minmax = minmax.transform(X_train_cp)
X_test_minmax = minmax.transform(X_test_cp)

In [46]:
# 2. 변환된 학습 데이터의 describe()를 출력하세요.
X_train_minmax_df = pd.DataFrame(X_train_minmax , columns=X.columns)
print(X_train_minmax_df.describe())

             preg        plas        pres        skin        insu        mass  \
count  614.000000  614.000000  614.000000  614.000000  614.000000  614.000000   
mean     0.224660    0.459238    0.491225    0.239591    0.168320    0.291374   
std      0.194950    0.209817    0.125256    0.096651    0.108045    0.139553   
min      0.000000    0.000000    0.000000    0.000000    0.000000    0.000000   
25%      0.058824    0.300699    0.408163    0.195652    0.144033    0.192740   
50%      0.176471    0.426573    0.489796    0.239130    0.150892    0.290389   
75%      0.352941    0.587413    0.571429    0.271739    0.157750    0.374233   
max      1.000000    1.000000    1.000000    1.000000    1.000000    1.000000   

             pedi         age  
count  614.000000  614.000000  
mean     0.175246    0.206107  
std      0.147127    0.197224  
min      0.000000    0.000000  
25%      0.071715    0.050000  
50%      0.132962    0.133333  
75%      0.247327    0.333333  
max      1.000

In [47]:
# 3. 변환된 데이터로 KNN(k=5)을 학습하세요.
model_minmax = KNeighborsClassifier(n_neighbors=5)
model_minmax.fit(X_train_minmax , y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [48]:
# 4. 학습 정확도와 테스트 정확도를 출력하세요.
print(model_minmax.score(X_train_minmax , y_train))
print(model_minmax.score(X_test_minmax , y_test))

0.8322475570032574
0.7337662337662337


In [49]:
# 5. 재현율을 출력하세요.
pred_minmax = model_minmax.predict(X_test_minmax)
print(recall_score(y_test , pred_minmax))

0.5555555555555556


In [50]:
# 6. 세 가지 전처리 방법의 성능을 비교하세요.
model_none = KNeighborsClassifier(n_neighbors=5)
model_none.fit(X_train_cp , y_train)
pred_none = model_none.predict(X_test_cp)

print(round(model_none.score(X_train_cp , y_train) , 2))
print(round(model_scaled.score(X_train_scaled , y_train) , 2))
print(round(model_minmax.score(X_train_minmax , y_train) , 2))
print()
print(round(model_none.score(X_test_cp , y_test) , 2))
print(round(model_scaled.score(X_test_scaled , y_test) , 2))
print(round(model_minmax.score(X_test_minmax , y_test) , 2))
print()
print(round(recall_score(y_test , pred_none) , 2))
print(round(recall_score(y_test , pred_scaled) , 2))
print(round(recall_score(y_test , pred_minmax) , 2))

0.8
0.83
0.83

0.68
0.75
0.73

0.5
0.61
0.56


In [51]:
data = {
    "KNN 전처리 방법" : ["스케일링 없음" , "StandardScaler" , "MinMaxScaler"] ,
    "학습 정확도" : [0.8 , 0.83 , 0.83] ,
    "테스트 정확도" : [0.68 , 0.75 , 0.73] ,
    "재현율" : [0.5 , 0.61 , 0.56]
}
compare_df = pd.DataFrame(data)
compare_df = compare_df.set_index("KNN 전처리 방법")
compare_df

,학습 정확도,테스트 정확도,재현율
KNN 전처리 방법,,,
스케일링 없음,0.80,0.68,0.50
StandardScaler,0.83,0.75,0.61
MinMaxScaler,0.83,0.73,0.56


**분석 질문**

1. 테스트 정확도가 가장 높은 전처리 방법은 무엇인가요?
2. 재현율이 가장 높은 전처리 방법은 무엇인가요?
3. 정확도가 높은 방법과 재현율이 높은 방법이 다를 수도 있나요?
4. 스케일링을 적용한다고 해서 정확도가 반드시 높아지는 것은 아닌 이유는 무엇일까요?

답
1. StandardScaler입니다.
2. StandardScaler입니다.
3. 다를 수 있습니다.
4. 모델이 이미 최적점을 찾은 상태라면 스케일링을 한다고 정확도가 더 올라가지 않을 수 있습니다.

### 10. 종합 — KNN 설정 선택 및 최종 분석

지금까지 실험한 결과를 이용하여 가장 적절하다고 생각하는 KNN 설정을 선택하세요.

**1단계 — 스케일링 방법 선택**

다음 중 하나를 선택하세요.

- 스케일링 없음
- StandardScaler
- MinMaxScaler

답 - StandardScaler 선택

**2단계 — k값 실험**

선택한 스케일링 방법으로 다음 k값을 실험하세요.

`1, 3, 5, 7, 10, 15, 20, 30`

각 k값에 대해 다음 결과를 표로 정리하세요.

- 학습 정확도
- 테스트 정확도
- 재현율

In [52]:
n_neighbors = [1 , 3 , 5 , 7 , 10 , 15 , 20 , 30]
for n in n_neighbors:
    model_scaled_n = KNeighborsClassifier(n_neighbors=n)
    model_scaled_n.fit(X_train_scaled , y_train)
    train_score_scaled_n = model_scaled_n.score(X_train_scaled , y_train)
    test_score_scaled_n = model_scaled_n.score(X_test_scaled , y_test)
    pred_scaled_n = model_scaled_n.predict(X_test_scaled)
    recall_score_scale_n = recall_score(y_test , pred_scaled_n)
    print(round(train_score_scaled_n , 2) , round(test_score_scaled_n , 2) , round(recall_score_scale_n , 2))

1.0 0.72 0.56
0.85 0.72 0.61
0.83 0.75 0.61
0.82 0.73 0.56
0.8 0.74 0.54
0.8 0.74 0.56
0.79 0.76 0.56
0.79 0.74 0.52


In [53]:
data = {
    "k값" : n_neighbors ,
    "학습 정확도" : [1.0 , 0.85 , 0.83 , 0.82 , 0.8 , 0.8 , 0.79 , 0.79] ,
    "테스트 정확도" : [0.72 , 0.72 , 0.75 , 0.73 , 0.74 , 0.74 , 0.76 , 0.74] ,
    "재현율" : [0.56 , 0.61 , 0.61 , 0.56 , 0.54 , 0.56 , 0.56 , 0.52] ,
}
result_df = pd.DataFrame(data)
result_df = result_df.set_index("k값")
result_df

,학습 정확도,테스트 정확도,재현율
k값,,,
1,1.00,0.72,0.56
3,0.85,0.72,0.61
5,0.83,0.75,0.61
7,0.82,0.73,0.56
10,0.80,0.74,0.54
15,0.80,0.74,0.56
20,0.79,0.76,0.56
30,0.79,0.74,0.52


**3단계 — 최종 설정 선택**

실험 결과를 비교하여 다음 두 가지를 결정하세요.

1. 사용할 스케일링 방법
2. 사용할 k값

단순히 정확도가 가장 높은 설정을 선택해도 되지만, 재현율이나 과대적합·과소적합 가능성을 고려하여 다른 설정을 선택해도 됩니다.

답
1. StandardScaler 선택
2. k = 5 선택

**4단계 — 선택한 설정으로 KNN 학습 및 평가**

선택한 설정으로 KNN을 학습하고 다음 결과를 출력하세요.

1. 학습 정확도
2. 테스트 정확도
3. 혼동행렬
4. 정밀도
5. 재현율
6. F1 점수

In [54]:
# 1. 학습 정확도
model_scaled_5 = KNeighborsClassifier(n_neighbors=5)
model_scaled_5.fit(X_train_scaled , y_train)
print(model_scaled_5.score(X_train_scaled , y_train))

0.8289902280130294


In [55]:
# 2. 테스트 정확도
print(model_scaled_5.score(X_test_scaled , y_test))

0.7532467532467533


In [56]:
# 3. 혼동행렬
pred_scaled_5 = model_scaled_5.predict(X_test_scaled)
print(confusion_matrix(y_test , pred_scaled_5))

[[83 17]
 [21 33]]


In [57]:
# 4. 정밀도
print(precision_score(y_test , pred_scaled_5))

0.66


In [58]:
# 5. 재현율
print(recall_score(y_test , pred_scaled_5))

0.6111111111111112


In [59]:
# 6. F1 점수
print(f1_score(y_test , pred_scaled_5))

0.6346153846153846


**5단계 — 최종 분석 보고서**

마크다운 셀에 다음 질문의 답변을 작성하세요.

1. 최종적으로 어떤 스케일링 방법과 k값을 선택했나요?
2. 해당 설정을 선택한 이유는 무엇인가요?
3. 스케일링 전후 KNN의 성능은 어떻게 달라졌나요?
4. k값이 달라지면서 정확도는 어떤 변화를 보였나요?
5. 정확도와 재현율 중 어떤 지표를 더 중요하게 고려했나요? 이유를 설명하세요.
6. 결측치 처리와 스케일링에서 학습 데이터에만 기준을 맞춰야 하는 이유는 무엇인가요?
7. 이번 과제를 진행하면서 머신러닝 전체 과정 중 가장 중요하다고 생각한 단계는 무엇인가요?

**참고:** 이번 과제는 아직 배우지 않은 검증 데이터·교차검증을 사용하지 않고, 동일한 테스트 데이터로 여러 설정의 결과를 비교합니다. 이는 수업 내용을 복습하기 위한 방식입니다. 실제 프로젝트에서는 설정 선택에 사용한 데이터와 최종 성능 평가에 사용할 테스트 데이터를 별도로 구분해야 합니다.

답
1. StandardScaler와 k = 5를 선택했습니다.
2. 정확도와 재현율 모두 상대적으로 높았기 때문입니다.
3. 정확도와 재현율 모두 상승하였습니다.
4. 조금씩 상승하다가 일정 값을 지나면 하락하였습니다.
5. 재현율을 더 중요하게 고려했습니다. 병에 대한 모델이므로 FN, 즉 실제로는 양성이나 음성으로 잘못 예측한 경우를 줄이는 게 더 중요하다고 판단했기 때문입니다.
6. 전체 데이터를 기준으로 할 경우 모델이 테스트 데이터에 대한 정보를 미리 알아버리기 때문입니다.
7. 데이터 전처리가 가장 중요하다고 생각합니다. 모델이 잘 정제된 데이터를 학습해야 그 예측을 신뢰할 수 있기 때문입니다.